# CSCI 7400 Assignment 7 by David Jean

## Read in data and check the head

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col

# Start Spark session
spark = SparkSession.builder \
    .appName("SentimentClassification") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

#Load the raw text file (
raw_df = spark.read.text("amazon_cells_labelled.txt")

# Split into sentence and label
df = raw_df.select(
    split(col("value"), "\t").getItem(0).alias("sentence"),
    split(col("value"), "\t").getItem(1).cast("int").alias("label")
)

# Show a few rows to confirm
df.show(5, truncate=False)


your 131072x1 screen size is bogus. expect trouble
25/04/03 20:21:24 WARN Utils: Your hostname, DESKTOP-BOJPASN resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/04/03 20:21:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/03 20:21:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/03 20:21:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+----------------------------------------------------------------------------------+-----+
|sentence                                                                          |label|
+----------------------------------------------------------------------------------+-----+
|So there is no way for me to plug it in here in the US unless I go by a converter.|0    |
|Good case, Excellent value.                                                       |1    |
|Great for the jawbone.                                                            |1    |
|Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!   |0    |
|The mic is great.                                                                 |1    |
+----------------------------------------------------------------------------------+-----+
only showing top 5 rows



## Check the data for imbalance in our target

In [2]:
from pyspark.sql.functions import col

# Count each class in the dataset
df.groupBy("label").count().orderBy("label").show()


+-----+-----+
|label|count|
+-----+-----+
|    0|  500|
|    1|  500|
+-----+-----+



## Preprocess the data by removing special characters, numbers, and other noise

In [ ]:
# Convert all sentences to lowercase and remove special characters and numbers
# Keeps only lowercase letters and spaces to simplify the text for processing

from pyspark.sql.functions import lower, regexp_replace

#Clean the text: lowercase + remove special chars/numbers
cleaned_df = df.select(
    lower(col("sentence")).alias("sentence"),
    col("label")
).withColumn(
    "sentence",
    regexp_replace("sentence", r"[^a-z\s]", "")  # keep only letters and spaces
)

cleaned_df.show(5, truncate=False)


+---------------------------------------------------------------------------------+-----+
|sentence                                                                         |label|
+---------------------------------------------------------------------------------+-----+
|so there is no way for me to plug it in here in the us unless i go by a converter|0    |
|good case excellent value                                                        |1    |
|great for the jawbone                                                            |1    |
|tied to charger for conversations lasting more than  minutesmajor problems       |0    |
|the mic is great                                                                 |1    |
+---------------------------------------------------------------------------------+-----+
only showing top 5 rows



## Tokenize the data; Remove stop words; Apply word2vec to convert tokens to numerical feature vectors. 

In [ ]:
# Tokenize the cleaned sentences into individual words using Spark's Tokenizer
# Adds a new column "words" containing tokenized lists for each sentence

from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="sentence", outputCol="words")
tokenized_df = tokenizer.transform(cleaned_df)

tokenized_df.select("sentence", "words", "label").show(5, truncate=False)


+---------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----+
|sentence                                                                         |words                                                                                                  |label|
+---------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+-----+
|so there is no way for me to plug it in here in the us unless i go by a converter|[so, there, is, no, way, for, me, to, plug, it, in, here, in, the, us, unless, i, go, by, a, converter]|0    |
|good case excellent value                                                        |[good, case, excellent, value]                                                                         |1    |
|great for the jawbone        

In [ ]:
# Remove common stop words (e.g., "the", "is", "and") from the tokenized word lists
# Outputs a new column "filtered" with stop words removed

from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="words", outputCol="filtered")
filtered_df = remover.transform(tokenized_df)

filtered_df.select("words", "filtered", "label").show(5, truncate=False)


+-------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+-----+
|words                                                                                                  |filtered                                                         |label|
+-------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------+-----+
|[so, there, is, no, way, for, me, to, plug, it, in, here, in, the, us, unless, i, go, by, a, converter]|[way, plug, us, unless, go, converter]                           |0    |
|[good, case, excellent, value]                                                                         |[good, case, excellent, value]                                   |1    |
|[great, for, the, jawbone]                                                                             |[grea

In [ ]:
# Convert filtered word tokens into numerical feature vectors using Word2Vec
# Trains a Word2Vec model and transforms the data into a "features" column for model input

from pyspark.ml.feature import Word2Vec

word2vec = Word2Vec(vectorSize=100, minCount=1, inputCol="filtered", outputCol="features")
model = word2vec.fit(filtered_df)
vectorized_df = model.transform(filtered_df)

# Final dataset ready for training
vectorized_df.select("filtered", "features", "label").show(5, truncate=False)


25/04/03 20:21:29 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


+-----------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Implement a classification algorithm of your choice using spark MLlib. (The data should be split into train and test), Train the model on the training dataset

- I experimented with a few different built in models, but basic LR ended up being the best without having to use something else besides Word2Vec. gradient boosting did not do well due to the small dataset.

In [7]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Split into train/test
train_df, test_df = vectorized_df.randomSplit([0.7, 0.3], seed=42)

#  Initialize base logistic regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    standardization=True
)

# Evaluator: Binary classifier using ROC AUC
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# Hyperparameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 0.02, 0.03]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.1]) \
    .build()


# CrossValidator
cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=8  # Use multiple cores
)

#  Train the model using CV
lr_model = cv.fit(train_df)

print("Best regParam:", lr_model.bestModel._java_obj.getRegParam())
print("Best elasticNetParam:", lr_model.bestModel._java_obj.getElasticNetParam())




Best regParam: 0.03
Best elasticNetParam: 0.0


### Model Training Summary

#### Train/Test Split
- Used a 70/30 split due to the small dataset (~1000 rows)
- 70% training ensures the model learns enough
- 30% testing gives reliable performance metrics

#### Evaluation Metric
- Used `areaUnderROC` (AUROC) for model selection
- Measures ranking quality of predictions

#### Model: Logistic Regression
- Chosen for simplicity and strong performance on dense features (Word2Vec)
- `maxIter=50` ensures convergence
- `standardization=True` keeps features on a comparable scale

#### Hyperparameter Grid
- Tuned `regParam` with values: [0.01, 0.1, 0.02, 0.03]
- Tuned `elasticNetParam`: [0.0, 0.1] to test L2 vs slight L1 mix
- Grid kept simple to avoid overfitting and reduce noise

#### Cross-Validation
- Used `numFolds=3` for balanced runtime and evaluation stability
- Cross-validation ensures results aren’t dependent on a single split

#### Parallelism
- Set `parallelism=8` to speed up training on a multi-core CPU


## Use the trained model to make predictions on the test dataset.

In [8]:
#  Full prediction output: sentence, true label, prediction, and probability
# Show full prediction output
predictions = lr_model.transform(test_df)
predictions.select("filtered", "label", "prediction", "probability").show(20, truncate=False)


+-------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|filtered                                                                                               |label|prediction|probability                             |
+-------------------------------------------------------------------------------------------------------+-----+----------+----------------------------------------+
|[, drain, weak, snap]                                                                                  |0    |1.0       |[0.4414526380503622,0.5585473619496377] |
|[, thumbs, seller]                                                                                     |1    |1.0       |[0.45448993151523304,0.5455100684847669]|
|[good, quality, bargain, bought, bought, cheapy, big, lots, sounded, awful, people, end, couldnt, hear]|1    |1.0       |[0.46195905463848536,0.5380409453615147]|
|[lot, websites,

# The assignment should also include model evaluation including accuracy, precision, recall, and F1 score. 

In [9]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initialize evaluators
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Compute metrics
accuracy = evaluator_accuracy.evaluate(predictions)
precision = evaluator_precision.evaluate(predictions)
recall = evaluator_recall.evaluate(predictions)
f1 = evaluator_f1.evaluate(predictions)

# Display results
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


Accuracy:  0.7148
Precision: 0.7145
Recall:    0.7148
F1 Score:  0.7146


In [10]:
from pyspark.sql.functions import col

# Group by actual and predicted label
confusion_df = predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
confusion_df.show()


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  101|
|    0|       1.0|   35|
|    1|       0.0|   38|
|    1|       1.0|   82|
+-----+----------+-----+



###  Model Evaluation Summary

#### Metrics on Test Data
- **Accuracy:** 0.7148
- **Precision:** 0.7145
- **Recall:** 0.7148
- **F1 Score:** 0.7146

#### What These Mean for Review Classification
- **Accuracy** measures overall correctness. About 71.5% of reviews were correctly classified as positive or negative.
- **Precision** tells us how often predicted positive reviews were actually positive. This helps reduce false positives (e.g., incorrectly calling a negative review "positive").
- **Recall** shows how many actual positive reviews were caught. This is important to avoid missing positive feedback that might indicate customer satisfaction.
- **F1 Score** balances precision and recall — a good overall indicator when both false positives and false negatives matter.

#### Confusion Matrix

| Actual Label | Predicted Label | Count |
|--------------|------------------|-------|
| 0 (Negative) | 0 (Negative)     | 101   |
| 0 (Negative) | 1 (Positive)     | 35    |
| 1 (Positive) | 0 (Negative)     | 38    |
| 1 (Positive) | 1 (Positive)     | 82    |


#### Insights
- The model performs equally well at identifying both positive and negative reviews.
- Some misclassification exists in both directions (~35 false positives, ~38 false negatives), but it's balanced.
- These results are strong for a simple logistic regression model with Word2Vec features, and demonstrate that the model captures key sentiment patterns in review text.

#### Potential Improvements
- Use a larger and more diverse dataset to help the model generalize better.
- Try more expressive models like Gradient Boosted Trees (GBT) or XGBoost for capturing nonlinear relationships.
- Experiment with neural networks (e.g., feedforward, LSTM, or transformer-based models) for deeper semantic understanding.
- Use TF-IDF or pretrained embeddings (e.g., GloVe, FastText, or BERT) for more meaningful feature representations.


